# 🔍 Code Duplication Detection - Google Colab

This notebook runs the CodeDuplicationDetection project on Google Colab with GPU acceleration.

**Requirements:**
- Go to Runtime → Change runtime type → Select **T4 GPU**

---

## 1️⃣ GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU")

## 2️⃣ Clone Project & Install Dependencies

In [ ]:
!git clone https://github.com/tuncaycelikkanat/CodeDuplicationDetection.git
%cd CodeDuplicationDetection
print("✅ Project cloned successfully!")

In [ ]:
!pip install -q xgboost optuna rapidfuzz shap joblib

In [ ]:
# Verify project structure
!ls -la
print("\n--- Dataset check ---")
!ls data/ 2>/dev/null || echo '❌ data/ directory not found!'
!ls data/poj104/ 2>/dev/null | head -5 && echo '...' || echo '❌ data/poj104/ not found!'

## 3️⃣ Load Dataset (if not included in repo)

If the dataset is not part of the repository, upload it manually.

In [ ]:
# Upload dataset directly to Colab:
# from google.colab import files
# uploaded = files.upload()  # upload poj104.zip
# !mkdir -p data && unzip poj104.zip -d data/

## 4️⃣ Train Model (with GPU) 🚀

Available models:
- `xgboost` — XGBoost (Optuna-tuned, Best F1: 0.9854)
- `ensemble` — XGBoost + RandomForest + SVM voting ensemble
- `random_forest` — Random Forest
- `linear_svm` — Linear SVM with calibration
- `dl_model` — Deep Learning (PyTorch MLP)

In [ ]:
# XGBoost with GPU (Optuna-tuned parameters)
!python main.py --model xgboost --device cuda --pairs 800000

In [ ]:
# Ensemble with GPU
# !python main.py --model ensemble --device cuda --pairs 800000

In [ ]:
# Deep Learning model with GPU
# !python main.py --model dl_model --device cuda --pairs 800000

In [ ]:
# Optuna hyperparameter tuning with GPU
# !python main.py --model xgboost --device cuda --pairs 800000 --tune --tune-trials 50

## 5️⃣ View Results

In [ ]:
import json
import os
from IPython.display import Image, display

# Find latest experiment
exp_dir = 'experiments'
if os.path.exists(exp_dir):
    exps = sorted([d for d in os.listdir(exp_dir) if d.startswith('exp_')])
    if exps:
        latest = os.path.join(exp_dir, exps[-1])
        print(f"📊 Latest experiment: {exps[-1]}\n")
        
        # Show metrics
        for split in ['train', 'test']:
            metrics_file = os.path.join(latest, f'metrics_{split}.json')
            if os.path.exists(metrics_file):
                with open(metrics_file) as f:
                    metrics = json.load(f)
                print(f"--- {split.upper()} ---")
                for k, v in metrics.items():
                    print(f"  {k}: {v:.4f}")
                print()
        
        # Show confusion matrix
        cm_file = os.path.join(latest, 'confusion_matrix_test.png')
        if os.path.exists(cm_file):
            display(Image(cm_file, width=400))
    else:
        print('No experiments found yet.')
else:
    print('experiments/ directory not found.')